<a href="https://colab.research.google.com/github/AristidesAntonioOrellanaZelaya/etl-proyecto-bi/blob/main/Notebooks/Hoteles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Cargar archivo
df_hoteles = pd.read_csv("https://raw.githubusercontent.com/AristidesAntonioOrellanaZelaya/etl-proyecto-bi/refs/heads/main/Data/fact_hoteles.csv")

print("Registros originales:", len(df_hoteles))

df_hoteles.head()

Registros originales: 5000


,id_reserva,fecha,departamento,tipo_hotel,habitaciones,ocupadas,tarifa_promedio
0,1,2024-09-07,Usulutan,Montaña,84,159,241
1,2,2021-06-06,San Miguel,Rural,195,61,53
2,3,2023-12-29,Santa Ana,Ciudad,178,48,54
3,4,2023-01-14,Santa Ana,Montaña,89,39,106
4,5,2024-06-22,La Paz,Ciudad,181,87,150


Revisar estructura

In [2]:
df_hoteles.info()

df_hoteles.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id_reserva       5000 non-null   int64 
 1   fecha            5000 non-null   object
 2   departamento     5000 non-null   object
 3   tipo_hotel       5000 non-null   object
 4   habitaciones     5000 non-null   int64 
 5   ocupadas         5000 non-null   int64 
 6   tarifa_promedio  5000 non-null   int64 
dtypes: int64(4), object(3)
memory usage: 273.6+ KB


,0
id_reserva,0
fecha,0
departamento,0
tipo_hotel,0
habitaciones,0
ocupadas,0
tarifa_promedio,0


**Transformar**

Convertir fecha

In [4]:
df_hoteles['fecha'] = pd.to_datetime(
    df_hoteles['fecha']
)

Crear atributos de tiempo

In [5]:
df_hoteles['anio'] = df_hoteles['fecha'].dt.year

df_hoteles['mes'] = df_hoteles['fecha'].dt.month

df_hoteles['dia'] = df_hoteles['fecha'].dt.day

df_hoteles['trimestre'] = df_hoteles['fecha'].dt.quarter

Limpiar departamento

In [6]:
df_hoteles['departamento'] = (
    df_hoteles['departamento']
    .astype(str)
    .str.strip()
    .str.title()
)

Limpiar tipo de hotel

In [7]:
df_hoteles['tipo_hotel'] = (
    df_hoteles['tipo_hotel']
    .astype(str)
    .str.strip()
    .str.title()
)

Eliminar duplicados

In [8]:
print(
    "Duplicados:",
    df_hoteles.duplicated().sum()
)

df_hoteles.drop_duplicates(inplace=True)

Duplicados: 0


Validar habitaciones

In [9]:
df_hoteles = df_hoteles[
    df_hoteles['habitaciones'] > 0
]

Validar ocupación

In [10]:
df_hoteles = df_hoteles[
    df_hoteles['ocupadas'] >= 0
]

df_hoteles = df_hoteles[
    df_hoteles['ocupadas']
    <= df_hoteles['habitaciones']
]

Validar tarifa

In [11]:
df_hoteles = df_hoteles[
    df_hoteles['tarifa_promedio'] > 0
]

Crear KPI principal: Ocupación Hotelera %

In [12]:
df_hoteles['porcentaje_ocupacion'] = (
    df_hoteles['ocupadas']
    / df_hoteles['habitaciones']
) * 100

Clasificar ocupación

In [13]:
df_hoteles['nivel_ocupacion'] = np.where(
    df_hoteles['porcentaje_ocupacion'] < 40,
    'Baja',
    np.where(
        df_hoteles['porcentaje_ocupacion'] < 70,
        'Media',
        'Alta'
    )
)

Ingreso potencial por día

In [14]:
df_hoteles['ingreso_estimado'] = (
    df_hoteles['ocupadas']
    * df_hoteles['tarifa_promedio']
).round(2)

Validación

In [15]:
print(df_hoteles.info())

df_hoteles.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 2919 entries, 1 to 4999
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_reserva            2919 non-null   int64         
 1   fecha                 2919 non-null   datetime64[ns]
 2   departamento          2919 non-null   object        
 3   tipo_hotel            2919 non-null   object        
 4   habitaciones          2919 non-null   int64         
 5   ocupadas              2919 non-null   int64         
 6   tarifa_promedio       2919 non-null   int64         
 7   anio                  2919 non-null   int32         
 8   mes                   2919 non-null   int32         
 9   dia                   2919 non-null   int32         
 10  trimestre             2919 non-null   int32         
 11  porcentaje_ocupacion  2919 non-null   float64       
 12  nivel_ocupacion       2919 non-null   object        
 13  ingreso_estimado      2

,id_reserva,fecha,habitaciones,ocupadas,tarifa_promedio,anio,mes,dia,trimestre,porcentaje_ocupacion,ingreso_estimado
count,2919.000000,2919,2919.000000,2919.000000,2919.000000,2919.000000,2919.000000,2919.000000,2919.000000,2919.000000,2919.000000
mean,2512.709832,2022-12-21 07:57:32.004110848,135.753340,71.075711,144.012676,2022.474135,6.492292,15.832134,2.501199,52.916779,10265.552929
min,2.000000,2021-01-01 00:00:00,20.000000,5.000000,35.000000,2021.000000,1.000000,1.000000,1.000000,2.512563,180.000000
25%,1260.500000,2021-12-03 00:00:00,105.000000,33.000000,91.000000,2021.000000,3.000000,8.000000,1.000000,29.970060,3696.000000
50%,2503.000000,2022-12-31 00:00:00,143.000000,65.000000,145.000000,2022.000000,7.000000,16.000000,3.000000,53.293413,7821.000000
75%,3767.000000,2023-12-28 00:00:00,174.000000,104.000000,198.000000,2023.000000,9.000000,23.000000,3.000000,76.190476,14914.500000
max,5000.000000,2024-12-29 00:00:00,199.000000,179.000000,249.000000,2024.000000,12.000000,31.000000,4.000000,100.000000,43318.000000
std,1446.793199,NaN,44.417934,44.633663,61.957437,1.133301,3.434274,8.797270,1.117152,27.200165,8318.151425


**Carga**

In [16]:
df_hoteles.to_csv(
    'dw_fact_hoteles.csv',
    index=False
)

print("ETL completado correctamente")

ETL completado correctamente
